# Customer Segmentation & RFM Analysis**Dataset:** UCI Online Retail II (`Year 2009-2010` sheet) — 525,461 transaction rows from a UK online giftware retailer.

**Question:** Who are our most valuable customers, who is slipping away, and where should retention effort go?

Load the data

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## 1. Load

In [2]:
# Kaggle mirror of UCI Online Retail II
import kagglehub
import os

path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")
data_file = os.path.join(path, 'online_retail_II.xlsx')

# Direct from UCI if Kaggle credentials are unavailable:
# data_file = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"

# NOTE: read_excel() takes the first sheet only ('Year 2009-2010').
# The file also contains 'Year 2010-2011'; this analysis is scoped to the first year.
df = pd.read_excel(data_file)
print(f"Rows: {len(df):,}")
display(df.head())

Using Colab cache for faster access to the 'online-retail-dataset' dataset.
Rows: 525,461


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Dates are shifted forward 14 years **for presentation only**, so the dashboard reads as a current period. No ordering or interval is altered. Recency is measured fromthe last transaction in the dataset, not from today.

In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
print(f"Original date range : {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
df['InvoiceDate'] = df['InvoiceDate'] + timedelta(days=365*14)
print(f"Shifted (+14y):{df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")

Original date range : 2023-11-28 to 2024-12-05
Shifted (+14y):2037-11-24 to 2038-12-02


## 3. Cleaning


In [4]:
anon= df['Customer ID'].isnull()
anon_rows = int(anon.sum())
anon_invoices= df.loc[anon, 'Invoice'].nunique()
anon_revenue = float((df.loc[anon, 'Quantity'] * df.loc[anon, 'Price']).sum())
print(f"Anonymous rows : {anon_rows:,} across {anon_invoices:,} invoices")
print(f"Anonymous revenue: GBP {anon_revenue:,.0f}  (real revenue, not attributable to a customer)")
print(f"v1 behaviour:one synthetic ID per ROW -> {anon_rows:,} fabricated customers")
df = df[~anon].copy()
df['Customer ID'] = df['Customer ID'].astype(int)
print(f"\nRows with an identified customer: {len(df):,}")
df = df[['Invoice','Quantity','InvoiceDate','Price','Customer ID','Country']].copy()
df['Amount'] = df['Quantity'] * df['Price']

# Returns and cancellations are EXCLUDED, not netted against purchases.
# This slightly overstates lifetime value for customers who returned goods.
returns     = int((df['Amount'] <= 0).sum())
returns_val = float(df.loc[df['Amount'] <= 0, 'Amount'].sum())
df = df[(df['Amount'] > 0) & (df['Quantity'] > 0)]
print(f"Removed {returns:,} return/cancellation rows (GBP {returns_val:,.0f})")
print(f"Clean shape: {df.shape}")

Anonymous rows : 107,927 across 5,229 invoices
Anonymous revenue: GBP 1,191,276  (real revenue, not attributable to a customer)
v1 behaviour:one synthetic ID per ROW -> 107,927 fabricated customers

Rows with an identified customer: 417,534
Removed 9,870 return/cancellation rows (GBP -483,795)
Clean shape: (407664, 7)


## 4. RFM metrics

In [5]:
analysis_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f"Reference date: {analysis_date}")

rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,   # Recency
    'Invoice'    : 'nunique',                                  # Frequency (distinct invoices)
    'Amount'     : 'sum'                                       # Monetary
}).reset_index()

rfm.columns = ['CustomerID','Recency','Frequency','Monetary']

print(f"\nCustomers: {len(rfm):,}   (v1 reported 108,214)")
print(f"Share with Frequency == 1: {(rfm.Frequency==1).mean():.1%}   (v1: ~95%)\n")
print(rfm[['Recency','Frequency','Monetary']].describe().round(2))

Reference date: 2024-12-06 20:01:00

Customers: 4,312   (v1 reported 108,214)
Share with Frequency == 1: 32.9%   (v1: ~95%)

       Recency  Frequency   Monetary
count  4312.00    4312.00    4312.00
mean     91.17       4.46    2048.24
std      96.86       8.17    8914.48
min       1.00       1.00       2.95
25%      18.00       1.00     307.99
50%      53.00       2.00     706.02
75%     136.00       5.00    1723.14
max     374.00     205.00  349164.35


## 5. Scoring

In [6]:
def f_score(freq):
    if freq >= 10: return 4
    if freq >= 5:  return 3
    if freq >= 2:  return 2
    return 1

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = rfm['Frequency'].apply(f_score)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm.R_Score.astype(str) + rfm.F_Score.astype(str) + rfm.M_Score.astype(str)

print("SANITY CHECK — mean actual Frequency by F_Score (must increase):")
print(rfm.groupby('F_Score').Frequency.agg(['count','mean','min','max']).round(2))

SANITY CHECK — mean actual Frequency by F_Score (must increase):
         count   mean  min  max
F_Score                        
1         1419   1.00    1    1
2         1713   2.75    2    4
3          776   6.49    5    9
4          404  19.94   10  205


## 6. Segmentation

In [7]:
def segment_customers(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if   r >= 4 and f >= 4 and m >= 4:  return 'Champions'
    elif r <= 2 and f >= 4 and m >= 4:  return "Can't Lose Them"      # before At Risk
    elif r <= 2 and f >= 3 and m >= 3:  return 'At Risk'
    elif r >= 3 and f >= 4:             return 'Loyal Customers'
    elif r >= 4 and 2 <= f <= 3:        return 'Potential Loyalists'
    elif r >= 4 and f == 1:             return 'New Customers'
    elif r == 1 and f <= 2:             return 'Lost'                 # before Hibernating
    elif r <= 2 and f <= 2:             return 'Hibernating'
    elif r <= 3 and f <= 3:             return 'About to Sleep'
    else:                               return 'Promising'

rfm['Segment'] = rfm.apply(segment_customers, axis=1)
print(f"Segments populated: {rfm.Segment.nunique()} of 10   (v1: 7)\n")
print(rfm.Segment.value_counts().to_string())

Segments populated: 10 of 10   (v1: 7)

Segment
Lost                   1032
About to Sleep         1028
Hibernating             861
Potential Loyalists     688
Champions               275
At Risk                 184
New Customers           117
Loyal Customers          95
Can't Lose Them          31
Promising                 1


## 7. Segment economics

In [8]:
insights = rfm.groupby('Segment').agg(
    Customers    =('CustomerID','count'),
    Avg_Recency  =('Recency','mean'),
    Avg_Frequency=('Frequency','mean'),
    Avg_Monetary =('Monetary','mean'),
    Total_Revenue=('Monetary','sum')
).round(2)

insights['Pct_Customers'] = (insights.Customers / len(rfm) * 100).round(2)
insights['Pct_Revenue']   = (insights.Total_Revenue / rfm.Monetary.sum() * 100).round(2)

print(insights.sort_values('Total_Revenue', ascending=False).to_string())

                     Customers  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  Pct_Customers  Pct_Revenue
Segment                                                                                                            
Champions                  275         6.92          22.39      13896.23     3821462.69           6.38        43.27
About to Sleep            1028        35.69           3.21       1223.16     1257407.71          23.84        14.24
Potential Loyalists        688         9.61           4.73       1745.65     1201006.11          15.96        13.60
Hibernating                861        83.90           2.08        801.97      690499.83          19.97         7.82
At Risk                    184       103.02           6.29       3060.40      563114.13           4.27         6.38
Loyal Customers             95        28.53          14.42       5803.51      551333.88           2.20         6.24
Lost                      1032       241.97           1.51        521.39

## 8. Revenue concentration

In [9]:
s = rfm.sort_values('Monetary', ascending=False)
cum = s.Monetary.cumsum() / s.Monetary.sum()
n80 = int((cum <= 0.80).sum()) + 1

print(f"Total customers     : {len(rfm):,}")
print(f"Total revenue       : GBP {rfm.Monetary.sum():,.0f}")
print(f"Avg customer value  : GBP {rfm.Monetary.mean():,.2f}   (v1 reported GBP 95.24)")
print(f"\n80% of revenue comes from the top {n80:,} customers ({n80/len(rfm)*100:.2f}%)")
print("v1 claimed 1.92% — an artifact of the inflated denominator, not a finding.")

Total customers     : 4,312
Total revenue       : GBP 8,832,003
Avg customer value  : GBP 2,048.24   (v1 reported GBP 95.24)

80% of revenue comes from the top 1,176 customers (27.27%)
v1 claimed 1.92% — an artifact of the inflated denominator, not a finding.


## 9. Export for the SQL / Power BI layer

In [10]:
rfm_export = rfm[['CustomerID','Recency','Frequency','Monetary', 'R_Score','F_Score','M_Score','RFM_Score','Segment']].copy()
rfm_export['Monetary'] = rfm_export['Monetary'].round(2)
rfm_export.to_csv('rfm_analysis_clean.csv', index=False)
print(f"Exported {len(rfm_export):,} customers to rfm_analysis_clean.csv")

Exported 4,312 customers to rfm_analysis_clean.csv


  ## Limitations
    
    1.Anonymous baskets are excluded.** 107,927 rows across 5,229 invoices (GBP 1,191,276) have no customer ID. That revenue is real but can't be tied to a customer, so it sits outside the segmentation. Every revenue figure above is identified-customer revenue.\n",
    2.Returns are excluded, not netted.** 9,870 return and cancellation rows (GBP -483,795) are dropped instead of being matched against the original purchases, which slightly overstates lifetime value for customers who returned goods.\n",
    3.Dates are shifted for presentation.** Gaps and ordering are unchanged, and recency is measured from the last date in the data.\n",
    4.only the 2009-2010 sheet is used.** The file also has 2010-2011, which would double the window and show repeat buying over a longer period.\n",
    5.The frequency bands are a judgement call** (1 / 2-4 / 5-9 / 10+). Quartiles can't split a count this skewed without breaking ties at random. Different bands would move some customers between segments, so the bands are stated here for anyone who wants to reproduce or challenge the result."